In [293]:
import pandas as pd
import importlib
import utils
import os
importlib.reload(utils)
importlib.reload(utils.data_processing)
os.chdir("C:/Users/Vale/Runalyze")

df = pd.read_csv("data/raw/training_data.csv", encoding="ISO-8859-1")

### Determine the general format of the dataset

The goal of this section is to get an initial understanding of the dataset's overall structure. This includes identifying whether the dataset contains any missing or duplicate values and reviewing its general shape.

In [294]:
df.head(5)

,Workout ID,Date,Distance (km),Duration (min),Calories (kcal),Average Pace (min/km),Average Speed (km/h),Max Speed (km/h),Elevation Gain (m),Elevation Loss (m),Start Time,Temperature (C),Wind Speed (km/h),Humidity (%)
0,1,14/06/2023,4.37,20:00,316,04:35,13.1,18.1,13,13.0,19:11,22.0,11.0,80.0
1,2,19/06/2023,4.45,20:00,315,04:29,13.3,15.5,12,13.0,20:06,29.0,14.0,53.0
2,3,01/07/2023,5.26,25:01,395,04:45,12.6,15.1,12,14.0,12:09,22.0,8.0,79.0
3,4,21/08/2023,4.03,20:02,312,04:58,12.1,14.3,0,10.0,19:03,34.0,8.0,42.0
4,5,04/10/2023,3.76,20:01,305,05:19,11.3,13.6,9,13.0,17:03,25.0,13.0,68.0


In [295]:
print("\nDataset Information:")
df.info()


Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46 entries, 0 to 45
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Workout ID             46 non-null     int64  
 1   Date                   46 non-null     object 
 2   Distance (km)          46 non-null     float64
 3   Duration (min)         46 non-null     object 
 4   Calories (kcal)        46 non-null     int64  
 5   Average Pace (min/km)  46 non-null     object 
 6   Average Speed (km/h)   46 non-null     float64
 7   Max Speed (km/h)       46 non-null     float64
 8   Elevation Gain (m)     44 non-null     object 
 9   Elevation Loss (m)     44 non-null     float64
 10  Start Time             45 non-null     object 
 11  Temperature (C)        45 non-null     float64
 12  Wind Speed (km/h)      41 non-null     float64
 13  Humidity (%)           41 non-null     float64
dtypes: float64(7), int64(2), object(5)
mem

In [296]:
print("\nMissing Values per column:")
df.isnull().sum()


Missing Values per column:


Workout ID               0
Date                     0
Distance (km)            0
Duration (min)           0
Calories (kcal)          0
Average Pace (min/km)    0
Average Speed (km/h)     0
Max Speed (km/h)         0
Elevation Gain (m)       2
Elevation Loss (m)       2
Start Time               1
Temperature (C)          1
Wind Speed (km/h)        5
Humidity (%)             5
dtype: int64

In [297]:
print(f"\nNumber of duplicate rows in the dataset: {df.duplicated().sum()}")


Number of duplicate rows in the dataset: 0


### How many rows and columns are in the dataset?

This section answers the question by inspecting the dataset's shape to understand its overall structure.

In [298]:
rows, cols = df.shape
print(f"The dataset contains {rows} rows and {cols} columns")

The dataset contains 46 rows and 14 columns


### What are the names and data types of each column?

In this section, we examine the dataset's structure by listing the column names and their corresponding data types.

In [299]:
df.dtypes

Workout ID                 int64
Date                      object
Distance (km)            float64
Duration (min)            object
Calories (kcal)            int64
Average Pace (min/km)     object
Average Speed (km/h)     float64
Max Speed (km/h)         float64
Elevation Gain (m)        object
Elevation Loss (m)       float64
Start Time                object
Temperature (C)          float64
Wind Speed (km/h)        float64
Humidity (%)             float64
dtype: object

### Analyze the distribution of numerical variables

This section aims to provide a statistical summary of all numerical columns in the dataset.

In [300]:
print("Statistical summary of numerical variables:")
df.loc[:, df.columns != "Workout ID"].describe()

Statistical summary of numerical variables:


,Distance (km),Calories (kcal),Average Speed (km/h),Max Speed (km/h),Elevation Loss (m),Temperature (C),Wind Speed (km/h),Humidity (%)
count,46.000000,46.000000,46.000000,46.000000,44.000000,45.000000,41.000000,41.000000
mean,3.876087,288.434783,13.915217,16.850000,14.045455,15.333333,11.000000,70.317073
std,2.014538,160.167642,1.570558,1.828873,6.884274,7.507572,2.924038,18.028365
min,1.000000,67.000000,11.100000,13.600000,0.000000,2.000000,5.000000,38.000000
25%,3.002500,228.500000,12.800000,15.425000,10.000000,9.000000,10.000000,61.000000
50%,4.045000,310.000000,13.550000,16.900000,13.000000,16.000000,10.000000,67.000000
75%,4.837500,316.750000,14.950000,17.775000,15.000000,21.000000,13.000000,86.000000
max,10.010000,811.000000,16.800000,21.200000,31.000000,34.000000,18.000000,100.000000


#### Observations from the Distribution Analysis

1. The column `Workout ID` is numerical but holds unique identifiers, which are not relevant for statistical analysis.
2. The `Duration (min)` column is in string format (object), preventing inclusion in numerical analyses. 
   Future Action: Convert this column to seconds for uniformity and analysis.
   
3. The `Elevation Gain` column, although numerical, is stored as an object and should be converted to an integer.

In [301]:
medians = df.median(numeric_only=True)
print("Medians:\n", medians)

Medians:
 Workout ID               23.500
Distance (km)             4.045
Calories (kcal)         310.000
Average Speed (km/h)     13.550
Max Speed (km/h)         16.900
Elevation Loss (m)       13.000
Temperature (C)          16.000
Wind Speed (km/h)        10.000
Humidity (%)             67.000
dtype: float64


#### Data cleaning: Conversion of 'Duration (min)'

In [302]:
import os 
os.chdir(r"C:\Users\Vale\Runalyze")
from utils.data_processing import duration_to_seconds

df['Duration (sec)'] = df.apply(
    lambda row: duration_to_seconds(row['Duration (min)'], workout_id=row['Workout ID']),
    axis=1
)

df.head(5)

Error converting duration '03.33' for workout ID 42: invalid literal for int() with base 10: '03.33'


,Workout ID,Date,Distance (km),Duration (min),Calories (kcal),Average Pace (min/km),Average Speed (km/h),Max Speed (km/h),Elevation Gain (m),Elevation Loss (m),Start Time,Temperature (C),Wind Speed (km/h),Humidity (%),Duration (sec)
0,1,14/06/2023,4.37,20:00,316,04:35,13.1,18.1,13,13.0,19:11,22.0,11.0,80.0,1200.0
1,2,19/06/2023,4.45,20:00,315,04:29,13.3,15.5,12,13.0,20:06,29.0,14.0,53.0,1200.0
2,3,01/07/2023,5.26,25:01,395,04:45,12.6,15.1,12,14.0,12:09,22.0,8.0,79.0,1501.0
3,4,21/08/2023,4.03,20:02,312,04:58,12.1,14.3,0,10.0,19:03,34.0,8.0,42.0,1202.0
4,5,04/10/2023,3.76,20:01,305,05:19,11.3,13.6,9,13.0,17:03,25.0,13.0,68.0,1201.0


In [303]:
invalid_durations = df[df['Duration (sec)'].isnull()]
invalid_durations

,Workout ID,Date,Distance (km),Duration (min),Calories (kcal),Average Pace (min/km),Average Speed (km/h),Max Speed (km/h),Elevation Gain (m),Elevation Loss (m),Start Time,Temperature (C),Wind Speed (km/h),Humidity (%),Duration (sec)
41,42,23/11/2024,1.0,03.33,80,03.33,16.8,18.7,NaN,NaN,16:04,9.0,13.0,38.0,NaN


Identified an invalid duration format '03.33' for workout ID 42. Replace '.' with ':' to correct the format to '03:33'.

In [304]:
df['Duration (min)'] = df['Duration (min)'].str.replace(".", ":", regex=False)

df['Duration (sec)'] = df.apply(
    lambda row: duration_to_seconds(row['Duration (min)'], workout_id=row['Workout ID']),
    axis=1
)

df.head(5)

,Workout ID,Date,Distance (km),Duration (min),Calories (kcal),Average Pace (min/km),Average Speed (km/h),Max Speed (km/h),Elevation Gain (m),Elevation Loss (m),Start Time,Temperature (C),Wind Speed (km/h),Humidity (%),Duration (sec)
0,1,14/06/2023,4.37,20:00,316,04:35,13.1,18.1,13,13.0,19:11,22.0,11.0,80.0,1200
1,2,19/06/2023,4.45,20:00,315,04:29,13.3,15.5,12,13.0,20:06,29.0,14.0,53.0,1200
2,3,01/07/2023,5.26,25:01,395,04:45,12.6,15.1,12,14.0,12:09,22.0,8.0,79.0,1501
3,4,21/08/2023,4.03,20:02,312,04:58,12.1,14.3,0,10.0,19:03,34.0,8.0,42.0,1202
4,5,04/10/2023,3.76,20:01,305,05:19,11.3,13.6,9,13.0,17:03,25.0,13.0,68.0,1201


#### Data cleaning: Conversion of 'Average Pace' to seconds

In [305]:
from utils.data_processing import pace_to_seconds

df['Average Pace (sec/km)'] = df.apply(
    lambda row: pace_to_seconds(row['Average Pace (min/km)'], workout_id=row['Workout ID']),
    axis=1
)

df.head(5)

Error converting pace '03.33' for workout ID 42: invalid literal for int() with base 10: '03.33'
Error converting pace '03.49' for workout ID 45: invalid literal for int() with base 10: '03.49'


,Workout ID,Date,Distance (km),Duration (min),Calories (kcal),Average Pace (min/km),Average Speed (km/h),Max Speed (km/h),Elevation Gain (m),Elevation Loss (m),Start Time,Temperature (C),Wind Speed (km/h),Humidity (%),Duration (sec),Average Pace (sec/km)
0,1,14/06/2023,4.37,20:00,316,04:35,13.1,18.1,13,13.0,19:11,22.0,11.0,80.0,1200,275.0
1,2,19/06/2023,4.45,20:00,315,04:29,13.3,15.5,12,13.0,20:06,29.0,14.0,53.0,1200,269.0
2,3,01/07/2023,5.26,25:01,395,04:45,12.6,15.1,12,14.0,12:09,22.0,8.0,79.0,1501,285.0
3,4,21/08/2023,4.03,20:02,312,04:58,12.1,14.3,0,10.0,19:03,34.0,8.0,42.0,1202,298.0
4,5,04/10/2023,3.76,20:01,305,05:19,11.3,13.6,9,13.0,17:03,25.0,13.0,68.0,1201,319.0


Identified errors in pace conversion: '03.33' and '03.49' are invalid formats for conversion.
Replace '.' with ':' to correct the format.

In [306]:
df['Average Pace (min/km)'] = df['Average Pace (min/km)'].str.replace(".", ":", regex=False)

df['Average Pace (sec/km)'] = df.apply(
    lambda row: pace_to_seconds(row['Average Pace (min/km)'], workout_id=row['Workout ID']),
    axis=1
)

df.head(5)

,Workout ID,Date,Distance (km),Duration (min),Calories (kcal),Average Pace (min/km),Average Speed (km/h),Max Speed (km/h),Elevation Gain (m),Elevation Loss (m),Start Time,Temperature (C),Wind Speed (km/h),Humidity (%),Duration (sec),Average Pace (sec/km)
0,1,14/06/2023,4.37,20:00,316,04:35,13.1,18.1,13,13.0,19:11,22.0,11.0,80.0,1200,275
1,2,19/06/2023,4.45,20:00,315,04:29,13.3,15.5,12,13.0,20:06,29.0,14.0,53.0,1200,269
2,3,01/07/2023,5.26,25:01,395,04:45,12.6,15.1,12,14.0,12:09,22.0,8.0,79.0,1501,285
3,4,21/08/2023,4.03,20:02,312,04:58,12.1,14.3,0,10.0,19:03,34.0,8.0,42.0,1202,298
4,5,04/10/2023,3.76,20:01,305,05:19,11.3,13.6,9,13.0,17:03,25.0,13.0,68.0,1201,319


#### Data cleaning: Conversion of 'Elevation Gain'

In [307]:
df['Elevation Gain (m)'].head(5)

0    13
1    12
2    12
3     0
4     9
Name: Elevation Gain (m), dtype: object

In [308]:
non_numeric = df[~df['Elevation Gain (m)'].astype(str).str.isdigit()]

rows = non_numeric[~non_numeric['Elevation Gain (m)'].isna()]
rows

,Workout ID,Date,Distance (km),Duration (min),Calories (kcal),Average Pace (min/km),Average Speed (km/h),Max Speed (km/h),Elevation Gain (m),Elevation Loss (m),Start Time,Temperature (C),Wind Speed (km/h),Humidity (%),Duration (sec),Average Pace (sec/km)
34,35,02/11/2024,1.02,03:49,83,03:45,16.0,16.8,16:24,20.0,NaN,NaN,NaN,NaN,229,225


#### Handling Erroneous Value '16:24'

Identified an incorrect entry ('16:24') in the `Elevation Gain (m)` column.

In [309]:
def pace_to_seconds(pace_str):
    minutes, seconds = map(int, pace_str.split(":"))
    return minutes * 60 + seconds

# all the workouts with an incorrect format in 'Elevation Gain (m)'
erroneous_workouts = []


workout = {
    'Workout ID': rows['Workout ID'].values[0],
    'Distance (km)': rows['Distance (km)'].values[0],
    'Duration (sec)': rows['Duration (sec)'].values[0],
    'Average Pace (sec/km)': rows['Average Pace (sec/km)'].values[0],
    'Elevation Gain (m)': rows['Elevation Gain (m)'].values[0]
}

erroneous_workouts.append(workout)
erroneous_workouts

[{'Workout ID': 35,
  'Distance (km)': 1.02,
  'Duration (sec)': 229,
  'Average Pace (sec/km)': 225,
  'Elevation Gain (m)': '16:24'}]

In [310]:
for wk in erroneous_workouts:
    expected_duration = wk['Average Pace (sec/km)'] * wk['Distance (km)']
    
    print(f"Workout ID: {wk['Workout ID']}")
    print("---------------------------\n")
    print(f"Expected Duration (seconds): {expected_duration}\n")
    print(f"Actual Duration (seconds): {wk['Duration (sec)']}\n")

Workout ID: 35
---------------------------

Expected Duration (seconds): 229.5

Actual Duration (seconds): 229



**Mathematical Verification**

We used the formula: 
     $$
\text{Duration (sec)} = \text{Average Pace (sec/km)} \times \text{Distance (km)}
     $$
     
By calculating the expected duration based on the average pace and distance, we demonstrated that the value `'16:24'` does not match the expected duration.

For this reason, we'll move the value `'16:24'` to the appropriate column `Start Time` and removed it from the `Elevation Gain (m)` column.

In [311]:
from utils.data_processing import is_valid_time_format

for wk in erroneous_workouts:
    if is_valid_time_format(wk['Elevation Gain (m)']):
        idx = int(wk['Workout ID'] - 1)
        df.loc[idx, 'Start Time'] = df.loc[idx, 'Elevation Gain (m)']
        df.loc[idx, 'Elevation Gain (m)'] = None
        print("Value successfully moved to 'Start Time'.")
    else:
        print(f"Value is not in a valid time format.")

Value successfully moved to 'Start Time'.


In [318]:
df['Elevation Gain (m)'] = df['Elevation Gain (m)'].apply(lambda x: int(x) if pd.notnull(x) else x)

In [319]:
df.dtypes

Workout ID                 int64
Date                      object
Distance (km)            float64
Duration (min)            object
Calories (kcal)            int64
Average Pace (min/km)     object
Average Speed (km/h)     float64
Max Speed (km/h)         float64
Elevation Gain (m)       float64
Elevation Loss (m)       float64
Start Time                object
Temperature (C)          float64
Wind Speed (km/h)        float64
Humidity (%)             float64
Duration (sec)             int64
Average Pace (sec/km)      int64
dtype: object

In [320]:
df.isnull().sum()

Workout ID               0
Date                     0
Distance (km)            0
Duration (min)           0
Calories (kcal)          0
Average Pace (min/km)    0
Average Speed (km/h)     0
Max Speed (km/h)         0
Elevation Gain (m)       3
Elevation Loss (m)       2
Start Time               0
Temperature (C)          1
Wind Speed (km/h)        5
Humidity (%)             5
Duration (sec)           0
Average Pace (sec/km)    0
dtype: int64

### Handling Missing Data

Identifying and addressing missing values to ensure a clean and consistent dataset.

#### Handling Missing Data in 'Elevation Gain (m)' column

In [334]:
missing_elevation_gain = df[df['Elevation Gain (m)'].isnull()]
missing_elevation_gain

,Workout ID,Date,Distance (km),Duration (min),Calories (kcal),Average Pace (min/km),Average Speed (km/h),Max Speed (km/h),Elevation Gain (m),Elevation Loss (m),Start Time,Temperature (C),Wind Speed (km/h),Humidity (%),Duration (sec),Average Pace (sec/km)
34,35,02/11/2024,1.02,03:49,83,03:45,16.0,16.8,NaN,20.0,16:24,NaN,NaN,NaN,229,225
41,42,23/11/2024,1.00,03:33,80,03:33,16.8,18.7,NaN,NaN,16:04,9.0,13.0,38.0,213,213
42,43,23/11/2024,1.00,03:38,79,03:38,16.5,19.5,NaN,NaN,16.11,9.0,NaN,NaN,218,218


In [336]:
for idx, row in missing_elevation_gain.iterrows():
    same_date_rows = df[df['Date'] == row['Date']]
    
    valid_elevation_rows = same_date_rows[same_date_rows['Elevation Gain (m)'].notnull()]
    
    if not valid_elevation_rows.empty:
        mean_elevation = valid_elevation_rows['Elevation Gain (m)'].mean()
        
        # Access a single value for a row/column label pair
        df.at[idx, 'Elevation Gain (m)'] = mean_elevation
    else:
        df.at[index, 'Elevation Gain (m)'] = 0

In [338]:
df['Elevation Gain (m)'].isnull().sum()

0

Impute missing values in the 'Elevation Gain (m)' column by calculating the mean of the valid values for rows with the same date. If no valid values exist, assign 0 to the missing values.

In [340]:
df.isnull().sum()

Workout ID               0
Date                     0
Distance (km)            0
Duration (min)           0
Calories (kcal)          0
Average Pace (min/km)    0
Average Speed (km/h)     0
Max Speed (km/h)         0
Elevation Gain (m)       0
Elevation Loss (m)       2
Start Time               0
Temperature (C)          1
Wind Speed (km/h)        5
Humidity (%)             5
Duration (sec)           0
Average Pace (sec/km)    0
dtype: int64

#### Handling Missing Data in 'Elevation Loss (m)', 'Temperature', 'Wind Speed', 'Humidity' columns, by following the same logic

In [341]:
columns_to_impute = ['Elevation Loss (m)', 'Temperature (C)', 'Wind Speed (km/h)', 'Humidity (%)']

for column in columns_to_impute:
    df = impute_missing_values(df, column)
    
df.isnull().sum()

NameError: name 'impute_missing_values' is not defined